# AIR Homework 3 - Block Stacking

## Assignment Overview

In this assignment, your robot must stack multiple blocks in a specified order. The main challenge we will be addressing is ensuring the stability of the tower as blocks are stacked. We will use a combination of Monte Carlo Tree Search (MCTS) and Reinforcement Learning (RL) to achieve this goal. MCTS will simulate future placements of blocks and help us choose where to place the blocks to avoid tower collapses, while RL will help us learn a policy to guide the search and reduce our reliance on a simulator.

You will implement the following components:
1. **Continuous MCTS**: Implement MCTS from scratch, allowing for continuous actions and stochastic outcomes.
2. **Train an RL agent**: Train a low-performance Actor-Critic agent to learn a policy and value function for the block stacking task.
3. **Guided MCTS**: Enhance the MCTS implementation to use the policy and value function learned by the RL agent to guide the search process.

To complete the assignment, read through the task notebooks, fill in the missing pieces of code in the `task*.py` files, and select your parameters wisely to ensure the robot can successfully complete the task. You are responsible for testing and debugging your implementation.

## Submission Instructions

**FAILURE TO FOLLOW THESE INSTRUCTIONS MAY RESULT IN A PENALTY OR REJECTION OF YOUR SUBMISSION.**

You are required to submit a single zip file named HW3_\<YOUR_ID\>.zip, replacing \<YOUR_ID\> with your unique identifier. The zip file must contain the following files at its top level (not inside a folder):
- `task1.py` and `task2.py` with your code implementations.

Do not rename the task files.

⚠️ Warning: These will be the only files that will be examined - so any change you make to other files, including the notebooks, will not be considered.

All questions in the notebooks are purely for thought. You do not have to answer them.

Your code will undergo automated testing to ensure correctness and efficiency. In case of failure, you will receive feedback that is proportionate to the issues found. Please ensure your code is well-documented and follows best practices.

### Running on the Lab Robot (Optional)

We are giving students the option to run their code on the lab robot. The purpose of this is to provide hands-on experience with real-world robotics hardware, but this is completely optional. The code in this assignment is structured to be agnostic to the environment on which it runs. If you choose to run on the lab robot, please follow these steps:
1. Ensure your code is functioning correctly in the provided simulation environment.
2. Save your RL agent algorithm object to a file named `rl_agent.zip`. More information on how to do this can be found in `task2.ipynb`.
3. Set an appointment on one of the available time slots. Bring your code (`task1.py` and `task2.py`) and saved agent files to the lab on a USB drive.

## Setup

This assignment runs locally in the course's conda environment. If you haven't done so yet, create the environment from `environment.yml`, activate it, and launch Jupyter:

```bash
conda env create -f environment.yml
conda activate air
jupyter lab
```

The assignment is split into the following files:
- `start_here.ipynb` - this notebook, with the assignment overview, the submission instructions, and our working environment.
- `task1.ipynb` and `task2.ipynb` - one notebook for each task, which walks you through the task and runs your code. Task 1 covers MCTS, and Task 2 covers training an RL agent and using it to guide MCTS.
- `task1.py` and `task2.py` - one file for each task. All the code you need to write is in these files, marked with `#TODO` comments.
- `scene.py` - builds our simulation environment. This file is provided, do not change it.

Each task notebook imports these files as modules in its setup section (for example, `import task1 as t1`) and calls your code through them (for example, `t1.reset_env(model, data)`). Your changes are reloaded automatically, so after saving a file you only need to rerun the relevant cells.

The tasks build on each other, so complete them in order. Each task notebook starts with a setup section that loads our working environment and imports your code from the previous task. This way, you can run each notebook on its own.

In [ ]:
%load_ext autoreload
%autoreload 2

# set plot theme
import seaborn as sns
sns.set_theme()

# Set the EGL backend for MuJoCo on Linux (no effect on other operating systems)
import platform

if platform.system() == "Linux":
  %env MUJOCO_GL=egl

## Our Working Environment

Our environment consists of our usual table, and UR5e, this time with four flat blocks for the robot to stack. The specifications of the environment are as follows:
1. The robot is mounted on the side of the table (not on its edge as before).
2. The UR5e robot has been equipped with a two-finger gripper to facilitate the pick-and-place operation.
3. The UR5e robot now uses position actuation instead, i.e., we can directly set the joint positions of the robot.

In [ ]:
# The scene is defined in scene.py. The first import downloads the UR5e robot and
# 2F85 gripper MJCF files from DeepMind's MuJoCo Menagerie
from scene import data, gripper_path, model, robot_path
print(f"UR5e robot MJCF path: {robot_path}\n")
print(f"Gripper MJCF path: {gripper_path}\n")

import mujoco
import mediapy as media

# Reset state and time to initial keyframe
mujoco.mj_resetDataKeyframe(model, data, model.key('robot_home').id)

duration= 1.0 # seconds
framerate = 60  # frames per second

frames = []
with mujoco.Renderer(model) as renderer:  # initialize renderer
    while data.time < duration:  # simulate for the specified duration

        # simulate one time step
        mujoco.mj_step(model, data)

        # capture frame at the specified framerate
        if len(frames) < data.time * framerate:

            # update scene info according to current state for specific camera
            renderer.update_scene(data, camera="frontview")

            # render the scene
            pixels = renderer.render()
            frames.append(pixels)

# show the captured video within the notebook
media.show_video(frames, fps=framerate)

**Next:** continue to Task 1 in `task1.ipynb`.